## nlp — maintenance log analysis
--
- NLP is used as a separate module from the machine failure prediction ML model.
- The NLP module takes maintenance logs as text input.
- It extracts important information such as **issue, component, and severity**.
- The extracted information will later be used by the decision and recommendation engine.

In [2]:
import pandas as pd
import numpy as np
import nltk

# Using a Synthetic Dataset for this because i couldn't find specific data required

In [4]:
df=pd.read_csv("/kaggle/input/datasets/nakulhemantkarpe/sentinel-ops-nlp-logs/sentinelops_nlp_maintenance_logs.csv")

In [5]:
df.head()

,log_id,log_date,machine_id,technician_id,log_text,component,issue,severity
0,1,2025-03-07,M-037,TECH-01,Maintenance check revealed random error code a...,PLC,random error code,Low
1,2,2025-05-21,M-046,TECH-04,Machine is showing slight noise during operati...,bearing,slight noise during operation,Low
2,3,2025-01-14,M-051,TECH-08,Operator reported routine wear observed origin...,filter,routine wear observed,Medium
3,4,2025-05-26,M-023,TECH-05,Ventilation duct showing signs of fan not spin...,ventilation duct,fan not spinning at rated speed,High
4,5,2025-11-20,M-023,TECH-14,Cooling fan inspection found rising process te...,cooling fan,rising process temperature,Medium


In [8]:
df.isnull().sum()

log_id           0
log_date         0
machine_id       0
technician_id    0
log_text         0
component        0
issue            0
severity         0
dtype: int64

In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   log_id         2000 non-null   int64 
 1   log_date       2000 non-null   object
 2   machine_id     2000 non-null   object
 3   technician_id  2000 non-null   object
 4   log_text       2000 non-null   object
 5   component      2000 non-null   object
 6   issue          2000 non-null   object
 7   severity       2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


Data is clean, free of null values
--


In [12]:
df.nunique()

log_id           2000
log_date          361
machine_id         60
technician_id      15
log_text         1807
component          32
issue              33
severity            3
dtype: int64

it has 1800 unique text logs and over 32,33 diffrent categories for component for issue, it makes this dataset fairly good
--

# Text Preprocessing
Lets import libraries and start preprocessing
--

In [55]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score

In [17]:
stop_words=set(stopwords.words("english"))
lemmatizer=WordNetLemmatizer()

Stopwords loaded and lemmatizer ready
--

In [20]:
def preprocess(text):
    words=word_tokenize(text.lower())
    clean_words=[]
    
    for i in words:
        if i.isalpha():
            clean_words.append(i)
    
    final_words=[]
    
    for i in clean_words:
        if i not in stop_words:
            final_words.append(lemmatizer.lemmatize(i))
    
    return " ".join(final_words)

kept all clean words first and then removed stop words, lemmatized final words and made a single string for them
--

In [22]:
df["processed_text"]=df["log_text"].apply(preprocess)

In [23]:
df[["log_text","processed_text"]].head()

,log_text,processed_text
0,Maintenance check revealed random error code a...,maintenance check revealed random error code a...
1,Machine is showing slight noise during operati...,machine showing slight noise operation bearing...
2,Operator reported routine wear observed origin...,operator reported routine wear observed origin...
3,Ventilation duct showing signs of fan not spin...,ventilation duct showing sign fan spinning rat...
4,Cooling fan inspection found rising process te...,cooling fan inspection found rising process te...


# Model training
lets split the data and train our predictive models
--


In [32]:
x=df["processed_text"]
targets={
    "component":df["component"],
    "issue":df["issue"],
    "severity":df["severity"]
}

X, and our y targets  defined,lets do train test split
--


In [35]:
train,test=train_test_split(df,test_size = 0.2,random_state=42)

In [36]:
x_train=train["processed_text"]
x_test=test["processed_text"]

y_train=train[["component","issue","severity"]]
y_test=test[["component","issue","severity"]]

passed the whole data first and then made x,y train and test sections
--

In [37]:
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

(1600,)
(400,)
(1600, 3)
(400, 3)




data is prepared, lets vectorize them using TF-IDF
--

In [62]:
vectorizer=TfidfVectorizer(ngram_range=(1,2))

In [63]:
x_train_tfidf=vectorizer.fit_transform(x_train)

In [64]:
x_test_tfidf=vectorizer.transform(x_test)

In [65]:
print(x_train_tfidf.shape)
print(x_test_tfidf.shape)

(1600, 1577)
(400, 1577)


Lets Train classifers now
--

In [66]:
models={}

for i in y_train.columns:
    models[i]=LogisticRegression(max_iter=1000)

3 models for 3 columns
--

In [73]:
for i in models:
  models[i].fit(x_train_tfidf,y_train[i])

In [74]:
predictions={}

for i in models:
    predictions[i]=models[i].predict(x_test_tfidf)

In [75]:
for i in models:
    print("-----",i,"-----")
    print("Accuracy:",accuracy_score(y_test[i],predictions[i]))
    print("Precision:",precision_score(y_test[i],predictions[i],average="weighted"))
    print("Recall:",recall_score(y_test[i],predictions[i],average="weighted"))
    print("F1-score:",f1_score(y_test[i],predictions[i],average="weighted"))

----- component -----
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0
----- issue -----
Accuracy: 1.0
Precision: 1.0
Recall: 1.0
F1-score: 1.0
----- severity -----
Accuracy: 0.6525
Precision: 0.5859531420358807
Recall: 0.6525
F1-score: 0.5869859150927337


In [77]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test["severity"],predictions["severity"]))

[[156   2  10]
 [  2  91  14]
 [ 78  33  14]]


Medium severity is a problem
--

 # severity classifier — baseline experiment

- A separate Logistic Regression classifier was initially tested for severity prediction.
- The model achieved a weighted F1-score of 58.88%.
- The results showed significant confusion between Medium and the other severity classes.
- This approach was not selected for the final system.
- Severity is instead determined using a rule-based policy after NLP extracts the issue and component.

In [80]:
HIGH_SEVERITY_KEYWORDS=[
    "excessive","seizure","beyond threshold","beyond rated limit",
    "power failure","sudden power loss","overheating","stalling",
    "deformation","cracking","overstrain","abnormal current",
    "no repeatable cause"
]

MEDIUM_SEVERITY_KEYWORDS=[
    "wear","fluctuation","restriction","degradation","poor finish",
    "intermittent","signal loss","error code","rising process temperature"
]

LOW_SEVERITY_KEYWORDS=[
    "minor","routine","no issue found","monitored",
    "calibration check passed","lubrication topped up",
    "slight noise","scheduled inspection completed"
]

 high severity keywords
--
- Defined keywords that indicate critical or high-severity maintenance issues.
- Examples include overheating, seizure, deformation, overstrain, and power failure.
- These keywords are used by the rule-based severity engine.


 medium and low severity keywords
--
- Defined keywords for medium and low severity maintenance issues.
- Medium keywords represent conditions such as wear, degradation, restriction, and error codes.
- Low keywords represent minor or routine conditions such as monitored issues, slight noise, and completed inspections.

In [83]:
HIGH_CRITICALITY_COMPONENTS={
    "motor","power supply unit","drive controller","electrical panel",
    "inverter","main shaft","coupling"
}

LOW_CRITICALITY_COMPONENTS={
    "gasket","filter","lubrication line","belt"
}

component criticality rules
--
- Defined components that have higher or lower maintenance criticality.
- Critical components such as motor, inverter, and main shaft can increase severity.
- Components such as filter, belt, and lubrication line are treated as lower criticality.
- Component criticality is used when the issue text does not directly determine severity.

In [81]:
def classify_severity(issue,component):
    issue_l=issue.lower()

    for i in HIGH_SEVERITY_KEYWORDS:
        if i in issue_l:
            return "High"

    for i in LOW_SEVERITY_KEYWORDS:
        if i in issue_l:
            return "Low"

    for i in MEDIUM_SEVERITY_KEYWORDS:
        if i in issue_l:
            return "Medium"

    comp_l=component.lower()

    if comp_l in HIGH_CRITICALITY_COMPONENTS:
        return "High"

    if comp_l in LOW_CRITICALITY_COMPONENTS:
        return "Low"

    return "Medium"

 rule-based severity classification
--
- Created `classify_severity()` to determine severity from the predicted issue and component.
- Checks high-severity keywords first.
- Checks low-severity keywords next.
- Checks medium-severity keywords after that.
- If no issue keyword matches, component criticality is checked.
- Returns Medium as the default when no rule matches.

In [82]:
print(classify_severity("abnormal current draw","motor"))
print(classify_severity("routine wear observed","filter"))
print(classify_severity("random error code","PLC"))

High
Low
Medium


# why the severity classifier was not used in the final model

- Initially, severity was treated as a classification target and trained using Logistic Regression.
- The baseline achieved 66% accuracy and 58.88% weighted F1-score.
- The confusion matrix showed that Medium severity was frequently confused with High and Low.
- Severity in SentinelOps is treated as a policy-driven decision rather than an independent NLP prediction.
- Therefore, the final NLP module predicts only **issue** and **component**.
- Severity is determined afterward using `classify_severity()` based on the predicted issue and component.
- This avoids training an ML model to reproduce deterministic severity rules.

In [84]:
def nlp_predict(log):
    processed=preprocess(log)
    
    tfidf=vectorizer.transform([processed])
    
    issue=models["issue"].predict(tfidf)[0]
    component=models["component"].predict(tfidf)[0]
    
    severity=classify_severity(issue,component)
    
    return issue,component,severity

# final NLP prediction function
- Created one function to combine the complete NLP pipeline.
- Takes a new maintenance log as input.
- Applies the same NLTK preprocessing used during training.
- Converts the processed log into TF-IDF features using the trained vectorizer.
- Uses the trained issue and component models to make predictions.
- Passes the predicted issue and component to the rule-based severity classifier.
- Returns the predicted issue, component, and severity.

In [85]:
new_log="The motor is showing abnormal current draw during operation."

In [86]:
issue,component,severity=nlp_predict(new_log)

print("Issue:",issue)
print("Component:",component)
print("Severity:",severity)

Issue: abnormal current draw
Component: motor
Severity: High


In [87]:
test_logs=[
    "The motor is showing abnormal current draw during operation.",
    "Routine inspection found slight noise from the filter.",
    "The bearing is showing excessive vibration during operation.",
    "Cooling fan has rising process temperature."
]

In [88]:
for i in test_logs:
    issue,component,severity=nlp_predict(i)
    
    print("Log:",i)
    print("Issue:",issue)
    print("Component:",component)
    print("Severity:",severity)
    print()

Log: The motor is showing abnormal current draw during operation.
Issue: abnormal current draw
Component: motor
Severity: High

Log: Routine inspection found slight noise from the filter.
Issue: slight noise during operation
Component: filter
Severity: Low

Log: The bearing is showing excessive vibration during operation.
Issue: minor vibration noted
Component: bearing
Severity: Low

Log: Cooling fan has rising process temperature.
Issue: rising process temperature
Component: cooling fan
Severity: Medium



# final NLP module

- NLTK is used for text preprocessing.
- TF-IDF converts processed maintenance logs into numerical features.
- Logistic Regression predicts the maintenance issue and component.
- A rule-based engine determines severity using the predicted issue and component.
- The final NLP output is:
  - Issue
  - Component
  - Severity
- This output will later be passed to the Decision & Recommendation Engine.

# Lets Export now

In [89]:
import joblib

In [90]:
joblib.dump(vectorizer,"nlp_tfidf_vectorizer.pkl")

['nlp_tfidf_vectorizer.pkl']

In [92]:
joblib.dump(models["issue"],"nlp_issue_model.pkl")

['nlp_issue_model.pkl']

In [93]:
joblib.dump(models["component"],"nlp_component_model.pkl")

['nlp_component_model.pkl']